## Parameter Settings

In [1]:
DATA_ROOT_DIR = '/workspace/data'
VIDEOS_DIR = f'{DATA_ROOT_DIR}/videos'
annotation_path = f'{DATA_ROOT_DIR}/annotations.jsonl'

In [2]:
# read annotation file as pandas dataframe
import json
import pandas as pd

with open(annotation_path, 'r') as f:
    annotations = [json.loads(line) for line in f]

df_annotations = pd.DataFrame(annotations) 

# 折り返さずに表示する設定
pd.set_option('display.max_colwidth', 200)

print(df_annotations.head())

                           video_path           selected_class  \
0            wyzi9GNZFMU_0_0to121.mp4    Camera Motion Editing   
1  8rKYl1CdXCc_5_276to660_scene02.mp4  Instance Motion Editing   
2           1s9DER1bpm0_10_0to213.mp4         Quantity Editing   
3            DaUJkmBvTKM_2_0to150.mp4    Visual Effect Editing   
4            Xw9Zsc9A924_0_0to138.mp4        Attribute Editing   

   selected_subclass  \
0           Dolly in   
1       Human motion   
2           Increase   
3  Background Change   
4   Color adjustment   

                                                                                                                                                                                               instruction  
0                                                        Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.  
1  Modify the video so that the man raises his l

In [3]:
# df の selected_class のユニークな値を取得する
unique_classes = df_annotations['selected_class'].unique()
print(unique_classes)

['Camera Motion Editing' 'Instance Motion Editing' 'Quantity Editing'
 'Visual Effect Editing' 'Attribute Editing' 'Style Editing'
 'Instance Editing' 'Camera Angle Editing']


In [4]:
import re

def parse_instruction(inst: str):
    inst_l = inst.lower()

    # =========================
    # 1. ZOOM / DOLLY
    # =========================
    if any(k in inst_l for k in ["zoom", "dolly", "close up"]):
        target = None

        # ターゲット抽出
        m = re.search(r"(on|of)\s+(.*)", inst_l)
        if m:
            target = m.group(2)

        return {
            "type": "zoom",
            "target": target,
            "strength": 0.3
        }

    # =========================
    # 2. HIGH ANGLE
    # =========================
    if any(k in inst_l for k in ["high angle", "look down"]):
        return {
            "type": "camera_angle",
            "mode": "high"
        }

    # =========================
    # 3. LOW ANGLE
    # =========================
    if any(k in inst_l for k in ["low angle", "look up"]):
        return {
            "type": "camera_angle",
            "mode": "low"
        }

    # =========================
    # 4. ARC SHOT
    # =========================
    if "arc shot" in inst_l:
        return {
            "type": "arc",
            "radius": 0.2
        }

    # =========================
    # 5. SHIFT CAMERA
    # =========================
    if "shift camera" in inst_l:
        if "high" in inst_l:
            return {"type": "camera_angle", "mode": "high"}
        if "low" in inst_l:
            return {"type": "camera_angle", "mode": "low"}

    # =========================
    # 6. その他 → skip
    # =========================
    return {
        "type": "skip"
    }

In [5]:
# df の instruction を list化する
instructions = df_annotations['instruction'].tolist()
num_total = len(instructions)
num_skipped = 0
for inst in instructions:
    parsed = parse_instruction(inst)
    if parsed["type"] == "skip":
        num_skipped += 1
    print(f"Instruction: {inst}")
    print(f"Parsed: {parsed}")
    print("-" * 50)

print(f"Total instructions: {num_total}")
print(f"Skipped instructions: {num_skipped}")

Instruction: Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.
Parsed: {'type': 'zoom', 'target': None, 'strength': 0.3}
--------------------------------------------------
Instruction: Modify the video so that the man raises his left hand to wave towards the camera in a friendly manner. Preserve his identity, the suit he is wearing, and the dark blue stage background. Ensure the motion is natural and temporally consistent, with no flickering or artifacts around the hand and arm.
Parsed: {'type': 'skip'}
--------------------------------------------------
Instruction: Increase the number of exhausted animals in the scene by adding more rhinos and buffalos lying on the ground in the mid-ground and background areas. Ensure these additional animals match the lighting, shadows, and 3D animation style of the original subjects for a seamless integration. Maintain temporal consistency so that the new

In [6]:
def is_zoom(inst):
    inst = inst.lower()

    keywords = [
        "zoom", "dolly", "close up",
        "closer", "zoom in", "zoom out",
        "focus on", "enlarge",
        "make bigger", "move closer"
    ]

    return any(k in inst for k in keywords)


def is_angle(inst):
    inst = inst.lower()

    keywords = [
        "angle", "low angle", "high angle",
        "from below", "from above",
        "look up", "look down"
    ]

    return any(k in inst for k in keywords)


def is_camera_motion(inst):
    inst = inst.lower()

    keywords = [
        "pan", "shift", "move camera",
        "arc", "orbit"
    ]

    return any(k in inst for k in keywords)


def route_instruction(inst):
    if is_zoom(inst):
        return "zoom"

    if is_angle(inst):
        return "angle"

    if is_camera_motion(inst):
        return "camera"

    return "skip"

In [7]:
# df の instruction を list化する
instructions = df_annotations['instruction'].tolist()
num_total = len(instructions)
num_skipped = 0
for inst in instructions:
    route = route_instruction(inst)
    if route == "skip":
        num_skipped += 1
    print(f"Instruction: {inst}")
    print(f"Route: {route}")
    print("-" * 50)

print(f"Total instructions: {num_total}")
print(f"Skipped instructions: {num_skipped}")

Instruction: Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.
Route: zoom
--------------------------------------------------
Instruction: Modify the video so that the man raises his left hand to wave towards the camera in a friendly manner. Preserve his identity, the suit he is wearing, and the dark blue stage background. Ensure the motion is natural and temporally consistent, with no flickering or artifacts around the hand and arm.
Route: skip
--------------------------------------------------
Instruction: Increase the number of exhausted animals in the scene by adding more rhinos and buffalos lying on the ground in the mid-ground and background areas. Ensure these additional animals match the lighting, shadows, and 3D animation style of the original subjects for a seamless integration. Maintain temporal consistency so that the new instances remain stationary and properly layered within th

In [8]:
def is_zoom(inst):
    inst = inst.lower()

    keywords = [
        # 明示
        "zoom", "dolly", "close up",

        # 暗黙（重要）
        "focus on",
        "make larger",
        "make bigger",
        "bring closer",
        "closer",
        "highlight",
        "emphasize",
        "prominent"
    ]

    return any(k in inst for k in keywords)


def is_angle(inst):
    inst = inst.lower()

    keywords = [
        "low angle", "high angle",
        "look up", "look down",
        "from below", "from above",
        "view from below",
        "view from above"
    ]

    return any(k in inst for k in keywords)


def is_camera_motion(inst):
    inst = inst.lower()

    keywords = [
        "pan",
        "shift",
        "move camera",
        "change viewpoint",
        "move perspective",
        "arc",
        "orbit"
    ]

    return any(k in inst for k in keywords)

def route_instruction(inst):
    if is_zoom(inst):
        return "zoom"

    if is_angle(inst):
        return "angle"

    if is_camera_motion(inst):
        return "camera"

    return "skip"

In [9]:
# df の instruction を list化する
instructions = df_annotations['instruction'].tolist()
num_total = len(instructions)
num_skipped = 0
for inst in instructions:
    route = route_instruction(inst)
    if route == "skip":
        num_skipped += 1
    print(f"Instruction: {inst}")
    print(f"Route: {route}")
    print("-" * 50)

print(f"Total instructions: {num_total}")
print(f"Skipped instructions: {num_skipped}")

Instruction: Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.
Route: zoom
--------------------------------------------------
Instruction: Modify the video so that the man raises his left hand to wave towards the camera in a friendly manner. Preserve his identity, the suit he is wearing, and the dark blue stage background. Ensure the motion is natural and temporally consistent, with no flickering or artifacts around the hand and arm.
Route: skip
--------------------------------------------------
Instruction: Increase the number of exhausted animals in the scene by adding more rhinos and buffalos lying on the ground in the mid-ground and background areas. Ensure these additional animals match the lighting, shadows, and 3D animation style of the original subjects for a seamless integration. Maintain temporal consistency so that the new instances remain stationary and properly layered within th

In [10]:
def is_non_opencv(inst):
    inst = inst.lower()

    keywords = [
        "change",
        "replace",
        "add",
        "remove",
        "transform",
        "style",
        "color",
        "turn into",
        "apply",
        "increase number"
    ]

    return any(k in inst for k in keywords)

def route_instruction(inst):
    inst_l = inst.lower()

    # -------------------------
    # 0. OpenCV禁止（最重要）
    # -------------------------
    if is_non_opencv(inst_l):
        return "skip"

    # -------------------------
    # 1. zoom
    # -------------------------
    if any(k in inst_l for k in [
        "zoom", "dolly", "close up"
    ]):
        return "zoom"

    # -------------------------
    # 2. angle
    # -------------------------
    if any(k in inst_l for k in [
        "angle", "look up", "look down",
        "from below", "from above"
    ]):
        return "angle"

    # -------------------------
    # 3. camera motion
    # -------------------------
    if any(k in inst_l for k in [
        "pan", "arc", "orbit", "shift camera"
    ]):
        return "camera"

    return "skip"

In [11]:
# df の instruction を list化する
instructions = df_annotations['instruction'].tolist()
num_total = len(instructions)
num_skipped = 0
for inst in instructions:
    route = route_instruction(inst)
    if route == "skip":
        num_skipped += 1
    print(f"Instruction: {inst}")
    print(f"Route: {route}")
    print("-" * 50)

print(f"Total instructions: {num_total}")
print(f"Skipped instructions: {num_skipped}")

Instruction: Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.
Route: skip
--------------------------------------------------
Instruction: Modify the video so that the man raises his left hand to wave towards the camera in a friendly manner. Preserve his identity, the suit he is wearing, and the dark blue stage background. Ensure the motion is natural and temporally consistent, with no flickering or artifacts around the hand and arm.
Route: skip
--------------------------------------------------
Instruction: Increase the number of exhausted animals in the scene by adding more rhinos and buffalos lying on the ground in the mid-ground and background areas. Ensure these additional animals match the lighting, shadows, and 3D animation style of the original subjects for a seamless integration. Maintain temporal consistency so that the new instances remain stationary and properly layered within th

In [12]:
!sudo python3 -m pip install sentence-transformers scikit-learn

In [13]:
# sentence-transformers：軽量な意味ベクトル（embedding）を生成するためのライブラリ
from sentence_transformers import SentenceTransformer

# cosine類似度：文章同士の意味の近さを数値化するために使用
from sklearn.metrics.pairwise import cosine_similarity


# ============================================
# ① モデルロード
# ============================================

# 背景意図：
# ・自然言語の表現ゆれ（zoom / focus / closer など）を吸収するため
# ・ルールベースでは拾えない「意味」を扱うため
model = SentenceTransformer('all-MiniLM-L6-v2')


# ============================================
# ② 参照文（プロトタイプ）の定義
# ============================================

# 背景意図：
# ・「これはcamera系だ」という基準を明示する
# ・few-shotのように振る舞う（教師データの代わり）
camera_refs = [
    "zoom in on subject",          # ズーム系の典型
    "dolly in camera",             # dolly表現
    "zoom out",                    # 逆ズーム
    "low angle shot",              # カメラ角度
    "high angle shot",
    "camera pan",                  # 水平移動
    "camera movement",             # 抽象的なカメラ操作
]

# 背景意図：
# ・OpenCVでは不可能な処理を明確に分離する
# ・誤爆（color変更やstyleをzoom扱い）を防ぐ
non_camera_refs = [
    "change color of object",      # 色変更（生成系）
    "replace background",          # 背景生成
    "add object",                  # 追加
    "remove object",               # 削除
    "apply style",                 # スタイル変換
    "make person wave",            # 動作生成
]


# ============================================
# ③ 参照文をembedding化
# ============================================

# 背景意図：
# ・比較を高速化（毎回encodeしない）
# ・「意味空間」に変換しておく
camera_emb = model.encode(camera_refs)
non_camera_emb = model.encode(non_camera_refs)


# ============================================
# ④ 分類関数
# ============================================

def classify_instruction(inst: str):
    """
    instructionを
    ・camera（OpenCV処理）
    ・skip（それ以外）
    に分類する
    """

    # ----------------------------------------
    # 4-1. 入力文をembedding化
    # ----------------------------------------

    # 背景意図：
    # ・自然言語をベクトル化して意味比較可能にする
    emb = model.encode([inst])


    # ----------------------------------------
    # 4-2. cameraとの類似度を計算
    # ----------------------------------------

    # 背景意図：
    # ・「どれくらいcamera操作に近いか」を測る
    cam_score = cosine_similarity(emb, camera_emb).max()


    # ----------------------------------------
    # 4-3. non-cameraとの類似度を計算
    # ----------------------------------------

    # 背景意図：
    # ・誤爆防止（例：color changeをzoomと誤認）
    non_score = cosine_similarity(emb, non_camera_emb).max()


    # ----------------------------------------
    # 4-4. 判定ロジック
    # ----------------------------------------

    # 背景意図：
    # ・cameraの方が意味的に近い場合のみ採用
    # ・曖昧なものは安全にskipへ
    if cam_score > non_score:
        return "camera"

    return "skip"

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3542.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# df の instruction を list化する
instructions = df_annotations['instruction'].tolist()
num_total = len(instructions)
num_skipped = 0
for inst in instructions:
    route = classify_instruction(inst)
    if route == "skip":
        num_skipped += 1
    print(f"Instruction: {inst}")
    print(f"Route: {route}")
    print("-" * 50)

print(f"Total instructions: {num_total}")
print(f"Skipped instructions: {num_skipped}")

Instruction: Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.
Route: camera
--------------------------------------------------
Instruction: Modify the video so that the man raises his left hand to wave towards the camera in a friendly manner. Preserve his identity, the suit he is wearing, and the dark blue stage background. Ensure the motion is natural and temporally consistent, with no flickering or artifacts around the hand and arm.
Route: camera
--------------------------------------------------
Instruction: Increase the number of exhausted animals in the scene by adding more rhinos and buffalos lying on the ground in the mid-ground and background areas. Ensure these additional animals match the lighting, shadows, and 3D animation style of the original subjects for a seamless integration. Maintain temporal consistency so that the new instances remain stationary and properly layered withi

In [15]:
# sentence-transformers：事前学習済みの意味エンコーダ（文章→ベクトル変換）を使うため
from sentence_transformers import SentenceTransformer

# cosine_similarity：ベクトル同士の意味的な近さを数値化するため
from sklearn.metrics.pairwise import cosine_similarity


# ============================================
# ① モデルロード
# ============================================

# 背景意図：
# ・MiniLMより高精度な意味理解を行うため
# ・instructionの長文・曖昧表現に対応するため
# ・RTX3090のGPUを活用して精度を優先するため
model = SentenceTransformer(
    'BAAI/bge-large-en-v1.5',  # 高精度embeddingモデル
    device='cuda'              # GPUで高速推論
)


# ============================================
# ② タスク定義（分類カテゴリ）
# ============================================

# 背景意図：
# ・instructionを「編集タスク」に分解するため
# ・OpenCVで処理できるものとそうでないものを分離するため
# ・表現ゆれ（zoom / focus / closer）を吸収するため
task_refs = {

    # ------------------------
    # Zoom系（スケール変化）
    # ------------------------
    "zoom": [
        "zoom in on subject",      # 基本的なズームイン
        "dolly in camera",         # dolly表現（zoomと同義）
        "zoom out slowly",         # ズームアウト
        "close up shot"            # クローズアップ表現
    ],

    # ------------------------
    # Angle系（視点変化）
    # ------------------------
    "angle": [
        "low angle shot",          # 下から見上げる視点
        "high angle view",         # 上から見下ろす視点
        "look up at subject",      # 言い換え
        "look down perspective"
    ],

    # ------------------------
    # Camera移動
    # ------------------------
    "camera_move": [
        "camera pan left",         # 水平移動
        "arc shot around object",  # 円運動
        "orbit camera movement"    # 周回移動
    ],

    # ------------------------
    # オブジェクト編集（生成系）
    # ------------------------
    "object_edit": [
        "add object to scene",     # 追加
        "remove object",           # 削除
        "replace object",          # 置換
        "insert new item"
    ],

    # ------------------------
    # 色変更
    # ------------------------
    "color_edit": [
        "change color of object",  # 色変更
        "make it red",
        "adjust color"
    ],

    # ------------------------
    # スタイル変換
    # ------------------------
    "style": [
        "anime style",             # アニメ調
        "cyberpunk style",         # サイバーパンク
        "oil painting style",      # 絵画風
        "pixel art"
    ],

    # ------------------------
    # 動作生成
    # ------------------------
    "motion": [
        "person waves",            # 手を振る
        "head turns",              # 頭を動かす
        "smile appears",           # 表情変化
        "object moves"
    ],

    # ------------------------
    # 背景変更
    # ------------------------
    "background": [
        "replace background",      # 背景差し替え
        "change scene background",
        "new environment"
    ]
}


# ============================================
# ③ embedding事前計算
# ============================================

# 背景意図：
# ・毎回encodeすると遅いため事前にベクトル化
# ・推論時は類似度計算のみで高速化
task_emb = {
    k: model.encode(v, normalize_embeddings=True)  # 正規化でcosine安定化
    for k, v in task_refs.items()
}


# ============================================
# ④ 分類関数
# ============================================

def classify_task(inst: str):
    """
    instructionを最も近いタスクに分類する
    """

    # ------------------------
    # 4-1. 入力文のembedding
    # ------------------------

    # 背景意図：
    # ・自然言語を意味ベクトルに変換
    # ・文の長さや表現ゆれを吸収
    emb = model.encode([inst], normalize_embeddings=True)


    # ------------------------
    # 4-2. 各タスクとの類似度計算
    # ------------------------

    # 背景意図：
    # ・どのタスクに最も意味的に近いかを評価
    scores = {}

    for task, ref_emb in task_emb.items():

        # 背景意図：
        # ・各タスク内で最も近い例文を採用
        score = cosine_similarity(emb, ref_emb).max()

        scores[task] = score


    # ------------------------
    # 4-3. 最適タスク選択
    # ------------------------

    # 背景意図：
    # ・最大類似度を持つタスクを採用
    best_task = max(scores, key=scores.get)

    return best_task, scores


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2491.56it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
# df の instruction を list化する
instructions = df_annotations['instruction'].tolist()
num_total = len(instructions)
num_skipped = 0
for inst in instructions:
    route, _ = classify_task(inst)
    if route == "skip":
        num_skipped += 1
    print(f"Instruction: {inst}")
    print(f"Route: {route}")
    print("-" * 50)

print(f"Total instructions: {num_total}")
print(f"Skipped instructions: {num_skipped}")

Instruction: Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.
Route: zoom
--------------------------------------------------
Instruction: Modify the video so that the man raises his left hand to wave towards the camera in a friendly manner. Preserve his identity, the suit he is wearing, and the dark blue stage background. Ensure the motion is natural and temporally consistent, with no flickering or artifacts around the hand and arm.
Route: camera_move
--------------------------------------------------
Instruction: Increase the number of exhausted animals in the scene by adding more rhinos and buffalos lying on the ground in the mid-ground and background areas. Ensure these additional animals match the lighting, shadows, and 3D animation style of the original subjects for a seamless integration. Maintain temporal consistency so that the new instances remain stationary and properly layered wi

In [17]:
!sudo python3 -m pip install transformers accelerate sentencepiece

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json

# ==========================================
# モデルロード
# ==========================================
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ==========================================
# プロンプト（超重要）
# ==========================================
def build_prompt(instruction):
    """
    この関数の意味: instructionから必要なパラメータを抽出するためのプロンプトを構築する。
    背景意図:
    - LLMに対して「instructionを解析して、必要なパラメータを抽出する」というタスクを明確に伝えるため。
    - ルールベースでは難しい「意味の理解」をLLMに任せるための指示を与えるため。
    - 出力形式をJSONに限定することで、後続の処理で扱いやすくするため。
    """
    return f"""
            You are an expert video editing parser.

            Extract structured parameters from the instruction.

            Return ONLY valid JSON.

            Schema:
            {{
            "group": "camera | other",
            "task": "zoom_in | zoom_out | low_angle | high_angle | arc_shot | unknown",
            "target": "string",
            "target_kind": "face | person | object | scene | unknown",
            "motion_style": "smooth | gradual | steady | slow | none",
            "keep_centered": true | false
            }}

            Instruction:
            {instruction}

            JSON:
            """


# ==========================================
# 推論
# ==========================================
def parse_with_llm(instruction):

    prompt = build_prompt(instruction)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.0,   # 安定性重視
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # JSON部分だけ抽出
    try:
        json_str = text.split("JSON:")[-1].strip()
        parsed = json.loads(json_str)
    except:
        parsed = {"error": text}

    return parsed

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 85.56it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


In [19]:
# df の instruction を list化する
instructions = df_annotations['instruction'].tolist()
num_total = len(instructions)
num_skipped = 0
for inst in instructions:
    parsed = parse_with_llm(inst)
    if parsed.get("error"):
        num_skipped += 1
    print(f"Instruction: {inst}")
    print(f"Parsed: {parsed}")
    print("-" * 50)

print(f"Total instructions: {num_total}")
print(f"Skipped instructions: {num_skipped}")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Instruction: Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.
Parsed: {'group': 'camera', 'task': 'dolly_in', 'target': 'man', 'target_kind': 'person', 'motion_style': 'smooth', 'keep_centered': True}
--------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Instruction: Modify the video so that the man raises his left hand to wave towards the camera in a friendly manner. Preserve his identity, the suit he is wearing, and the dark blue stage background. Ensure the motion is natural and temporally consistent, with no flickering or artifacts around the hand and arm.
Parsed: {'group': 'camera', 'task': 'unknown', 'target': 'man', 'target_kind': 'person', 'motion_style': 'natural', 'keep_centered': True}
--------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Instruction: Create a smooth zoom-out effect that starts from the center of the blurry colorful background and gradually reveals more of the surrounding space. Maintain the soft, dreamy bokeh quality of the pink and orange spheres while ensuring the vibrant purple graphic elements and the turquoise logo in the foreground remain sharp and anchored. The motion should be steady and continuous throughout the sequence, avoiding any sudden jumps or flickering.
Parsed: {'group': 'other', 'task': 'zoom_out', 'target': 'background', 'target_kind': 'scene', 'motion_style': 'smooth', 'keep_centered': False}
--------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Instruction: Transform the video into a soft watercolor painting style, preserving the fluid motions of the makeup application.
Parsed: {'group': 'other', 'task': 'unknown', 'target': 'video', 'target_kind': 'scene', 'motion_style': 'smooth', 'keep_centered': False}
--------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Instruction: Replace the pink swirl frosting on top of the cupcake with a vibrant scoop of mint green frosting that has a creamy, textured surface. Maintain the original position and scale of the frosting so that the person's hand appears to be placing the decoration onto the new mint green surface naturally. Ensure that the blue wrapper with the cute face, the wooden table, and all the surrounding colorful sprinkles remain completely unchanged throughout the video. The edit must be temporally consistent, featuring clean edges where the hand interacts with the frosting and no visual flickering or artifacts.
Parsed: {'error': '\n            You are an expert video editing parser.\n\n            Extract structured parameters from the instruction.\n\n            Return ONLY valid JSON.\n\n            Schema:\n            {\n            "group": "camera | other",\n            "task": "zoom_in | zoom_out | low_angle | high_angle | arc_shot | unknown",\n            "target": "string",\n     

# 集計結果レポート

100件を集計して確認した。結論は、「一部は使えるが、現状のままでは安定した中間表現としては弱い」。

------------------------------------------------------------------------

## 1. 全体の通過率

まず、ログ上の総数は 100件、error 扱いは 13件 だった。つまり、json.loads
相当で通ったのは 87件。これはログ末尾の Total instructions: 100 と
Skipped instructions: 13 と一致する。

-   総件数: 100件\
-   error 件数: 13件\
-   非 error 件数: 87件\
-   完全に schema 準拠していた件数: 58件

したがって、100件中42件は何らかの問題あり。\
非 error 87件の中でも、29件は schema違反が残っている。

ここでの「完全準拠」は、あなたの schema どおりに\
group, task, target, target_kind, motion_style, keep_centered
の6キーを持ち、余計なキーがなく、値も許容集合内、型も一致、という基準で数えた。

------------------------------------------------------------------------

## 2. 壊れ方の内訳

error の 13件 は、主に次の壊れ方だった。

-   Explanation付きで JSON の後に説明文が続く: 複数件\
    例として、背景置換の instruction で Explanation: が付いている。

-   複数の Instruction / JSON が混入して出力が連結される: 複数件\
    例として、beanie→hat の instruction の後に別の low-angle
    の例が混入している。

-   複数 JSON を勝手に並べる: 複数件\
    例として split-screen の background 置換で JSON が複数並んでいる。

この13件は、意味理解以前に出力制御が破綻している。

------------------------------------------------------------------------

## 3. 非 error 87件の中の schema違反

87件のうち、29件に schema違反があった。内訳は次のとおり。

### 余計なキー

-   gradual が追加: 4件\
-   motion_style_detail が追加: 2件\
-   background が追加: 1件

### 必須キー欠落

-   keep_centered 欠落: 13件\
-   target 欠落: 1件\
-   target_kind 欠落: 1件

### 許容値違反

-   task='dolly_in': 3件\
-   task='replace': 1件\
-   task='color_correction': 1件\
-   group='scene': 1件\
-   motion_style='natural': 1件\
-   motion_style='rapid': 1件\
-   target_kind='person \| object': 1件\
-   target_kind='person, object': 1件

### 型違反

-   motion_style が文字列ではなく dict: 1件\
    例: {'style': 'smooth', 'speed': 'gradual'}

------------------------------------------------------------------------

## 4. 出力分布

error でない87件だけを見ると、分布はこうだった。

### group

-   other: 53件\
-   camera: 33件\
-   scene: 1件 ← schema違反

### task

-   unknown: 53件\
-   zoom_in: 11件\
-   low_angle: 8件\
-   arc_shot: 5件\
-   dolly_in: 3件 ← schema違反\
-   high_angle: 3件\
-   zoom_out: 2件\
-   replace: 1件 ← schema違反\
-   color_correction: 1件 ← schema違反

### target_kind

-   person: 34件\
-   scene: 29件\
-   object: 17件\
-   face: 4件\
-   person \| object: 1件 ← schema違反\
-   person, object: 1件 ← schema違反\
-   欠落: 1件

### motion_style

-   none: 33件\
-   steady: 24件\
-   smooth: 20件\
-   slow: 4件\
-   gradual: 3件\
-   natural: 1件 ← schema違反\
-   rapid: 1件 ← schema違反\
-   dict型: 1件 ← schema違反

### keep_centered

-   False: 41件\
-   True: 33件\
-   欠落: 13件

------------------------------------------------------------------------

## 5. 中身として見たときの問題

数だけでなく、質的にも問題が見える。

まず、task='unknown' が 53/87件。\
つまり、非 error の中でも 約61% が task を具体化できていない。

さらに、意味的に怪しいものが複数ある。

-   背景置換 instruction に対して task='arc_shot' が出る例がある。\
-   人物の頭を少し傾ける instruction に対して task='low_angle'
    が出る例がある。\
-   「zoom out」なのに group='other' で返している例がある。\
-   物体置換 instruction なのに target='man holding cookie'
    と人物側へ寄っている例がある。

つまり、JSONが出ていても正しいとは限らない。

------------------------------------------------------------------------

## 6. 良い点

悪い点ばかりではない。camera系の明示的 instruction は比較的強い。

-   zoom_in: 11件\
-   low_angle: 8件\
-   high_angle: 3件\
-   zoom_out: 2件\
-   arc_shot: 5件

特に、素直な zoom / low-angle はかなり拾えている。たとえば zoom-in や
low-angle の例はそのままそれらの task で出ている。

------------------------------------------------------------------------

## 7. 判定

数値でまとめるとこうなる。

-   100件中58件しか完全準拠していない\
-   100件中13件は出力崩壊\
-   100件中29件は JSON には見えるが schema違反\
-   非 error でも 53件が task='unknown'\
-   さらに、意味的な誤分類が複数確認できる

なので判定は、\
「試作としては進んでいるが、後段の自動パイプラインにそのまま渡せる品質ではない」。

------------------------------------------------------------------------

## 結論

今の状態を一言で言うと、\
意味抽出のヒントにはなるが、制御信号としては不安定。

次にやるべきことは明確で、まず
LLMの出力をもっと狭い中間表現に落とすこと。\
今のように task を最終ラベルまで自由生成させると、dolly_in / replace /
color_correction / arc_shot誤爆 のような逸脱が続く。


## Pronpt 修正
### 現在
```
Return ONLY valid JSON
```

### 改善案
```
Return ONLY JSON.
Do not include any explanation.
Do not include any text before or after JSON.
Use ONLY the allowed values.

If uncertain, use "unknown".
```